##### ML용 전체 테이블 생성

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG

from datetime import date

In [ ]:
today = date.today().strftime("%Y%m%d") #260101

GOLD = f"{CATALOG}.gold"

In [ ]:
# 공통 feature (반경 무관)
COMMON_COLS = """
    s.farm_id,
    s.reference_date,
    s.poultry_species,
    s.flock_size,
    s.farm_count_3km,
    s.label_infected,
    h.farm_bird_nearest_dist_km,
    h.within_migratory_bird_site_10km,
    e.infected_farm_count_3km,
    e.outbreak_count_5yr,
    w.humidity,
    w.min_temp_7d,
    w.precipitation_7d,
    w.wind_speed_avg_7d
"""

JOINS = f"""
FROM {GOLD}.farm_status_feature_daily s
LEFT JOIN {GOLD}.farm_bird_exposure_features_daily b
    ON s.farm_id = b.farm_id AND s.reference_date = b.reference_date
LEFT JOIN {GOLD}.farm_bird_habitat_daily h
    ON s.farm_id = h.farm_id
LEFT JOIN {GOLD}.farm_epidemic_features_daily e
    ON s.farm_id = e.farm_id AND s.reference_date = e.reference_date
LEFT JOIN {GOLD}.farm_weather_features_daily w
    ON s.farm_id = w.farm_id AND s.reference_date = w.reference_date
WHERE s.farm_id IS NOT NULL
"""

In [ ]:
# 전체 반경 테이블
all_bird_cols = ",\n".join([
    f"""    
    b.bird_obs_count_7d_{km}km,
    b.bird_distance_sum_7d_{km}km,
    b.duck_obs_count_7d_{km}km,
    b.goose_obs_count_7d_{km}km,
    b.bird_obs_count_30d_{km}km,
    b.bird_distance_sum_30d_{km}km,
    b.duck_obs_count_30d_{km}km,
    b.goose_obs_count_30d_{km}km"""
    for km in [1, 3, 5, 7, 10]
])
(
    spark.sql(f"""
        SELECT {COMMON_COLS},
            {all_bird_cols},
            b.species_count_7d_3km,
            b.species_count_30d_3km,
            b.dominant_species_30d_3km
        {JOINS}
    """)
    .writeTo(f"{GOLD}.ml_dataset_radius_all_{today}")
    .using("delta")
    .createOrReplace()
)
print("✅ ml_dataset_radius_all_{today} 완료")